# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [1]:
#Import necessary libraries
import os
import tempfile
import subprocess
from io import BytesIO
from pydub import AudioSegment
import time
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import anthropic
import google.generativeai as genai
from google.generativeai import types
import bs4
from IPython.display import Markdown, display, update_display
import requests
import json
import datetime
from ddgs import DDGS

In [2]:
#load the .env file
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
ipgeo_api_key = os.getenv('IPGEO_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

In [3]:
system_message = f'You are a helpful assistant.  If you need more information, try to search the internet.  If you do not know the answer to a question, please say so.  The current date is {datetime.datetime.today().strftime('%Y-%m-%d %H:%M:%S')}. You have access to tools but only use them if applicable. Response normally otherwise. Respond in Markdown.'
#print(system_message)
openai = OpenAI()
claude = anthropic.Anthropic()
genai.configure()

In [4]:
def get_lat_long_by_ip():
    #Get the lat/Long by IP address.
    lat_long = {}
    url = f'https://api.ipgeolocation.io/v2/ipgeo?apiKey={ipgeo_api_key}'
    request = requests.get(url)
    content = bs4.BeautifulSoup(request.content, "html.parser")
    content_json = json.loads(content.text)
    lat_long.update({'lat' : content_json['location']['latitude']})
    lat_long.update({'long' : content_json['location']['longitude']})

    return lat_long

def get_weather_forecast(date) -> str:
    """Get the weather forecast for a specific date"""
    try:
        # Get the lat/long value by IP
        lat_long = get_lat_long_by_ip()
        lat = lat_long.get('lat')
        long = lat_long.get('long')
        
        if not lat or not long:
            return "Error: Could not determine location from IP address"
        
        # Use the lat/long to get the forecast URL from the National Weather Service
        weather_url = f'https://api.weather.gov/points/{lat},{long}'
        request = requests.get(weather_url)
        
        # Check if the request was successful
        if request.status_code != 200:
            return f"Error: Unable to get weather data. Status code: {request.status_code}"
        
        # Check if response content is not empty
        if not request.content:
            return "Error: Empty response from weather service"
            
        try:
            content = request.json()  # Use .json() method instead of manual parsing
        except json.JSONDecodeError:
            return "Error: Invalid JSON response from weather service"
        
        # Check if the expected data structure exists
        if 'properties' not in content or 'forecast' not in content['properties']:
            return "Error: Unexpected response format from weather service"
            
        forecast_url = content['properties']['forecast']
        
        # Call the forecast URL and scrape out the forecast periods into a dictionary
        forecast_request = requests.get(forecast_url)
        
        if forecast_request.status_code != 200:
            return f"Error: Unable to get forecast data. Status code: {forecast_request.status_code}"
            
        if not forecast_request.content:
            return "Error: Empty forecast response"
            
        try:
            forecast = forecast_request.json()  # Use .json() method
        except json.JSONDecodeError:
            return "Error: Invalid JSON in forecast response"
        
        # Check forecast structure
        if 'properties' not in forecast or 'periods' not in forecast['properties']:
            return "Error: Unexpected forecast response format"
            
        forecast_periods = {}
        for p in forecast['properties']['periods']:
            if 'startTime' in p and 'detailedForecast' in p:
                forecast_periods[str(p['startTime'])[:10]] = p['detailedForecast']

        result = forecast_periods.get(date)
        if result:
            return result
        else:
            # If exact date not found, return available dates
            available_dates = list(forecast_periods.keys())
            return f"No forecast available for {date}. Available dates: {', '.join(available_dates)}"
            
    except Exception as e:
        return f'Error getting weather forecast: {str(e)}'

In [5]:
def search_duckduckgo(query: str, num_results: int = 5) -> dict:   
    try:
        ddgs = DDGS()
        data = ddgs.text(query = query, max_results = num_results)

        print(f'DEBUG: Called search_duckduckgo')
        results = []
        
        # Add related topics
        for d in data:
            if isinstance(d, dict) and 'body' in d:
                results.append({
                    'title': d['title'],
                    'snippet': d['body'],
                    'url': d['href'],
                    'source': 'DuckDuckGo Related'
                })
        
        return {
            'status': 'success',
            'query': query,
            'results': results,
            'total_results': len(results)
        }
        
    except Exception as e:
        return {
            'status': 'error',
            'query': query,
            'error': str(e),
            'results': []
        }

In [6]:
def play_audio(audio_segment):
    temp_dir = tempfile.gettempdir()
    temp_path = os.path.join(temp_dir, "temp_audio.mp3")
    try:
        audio_segment.export(temp_path, format="mp3")
        #time.sleep(3) # Student Dominic found that this was needed. You could also try commenting out to see if not needed on your PC
        subprocess.call([
            "ffplay",
            "-nodisp",
            "-autoexit",
            "-hide_banner",
            temp_path
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    finally:
        try:
            os.remove(temp_path)
        except Exception:
            pass
 
def talker(message):
    response = openai.audio.speech.create(
        model="tts-1",
        voice="onyx",  # Also, try replacing onyx with alloy
        input=message
    )
    audio_stream = BytesIO(response.content)
    audio = AudioSegment.from_file(audio_stream, format="mp3")
    play_audio(audio)

In [7]:
#search_duckduckgo('state of Texas')
#ddgs = DDGS()
#ddgs.text(query='sate of texas', max_results=5)

In [8]:
weather_tool_openai = {
    "name": "get_weather_forecast",
    "description": "Get the weather forecast for a specific date, use this to determine an answer to questions like 'will it rain tomorrow?'",
    "parameters": {
        "type": "object",
        "properties": {
            "date": {
                "type": "string",
                "description": "The forecast date",
            },
        },
        "required": ["date"]
    }
}

search_tool_openai = {
    "name": "search_duckduckgo",
    "description": "Search the internet for information on a given query",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up information for"
            },
            "num_results": {
                "type": "integer",
                "description": "Number of search results to return (default: 5)",
            }
        },
        "required": ["query"]
    }
}

weather_tool_claude = {
    "type": "custom",
    "name": "get_weather_forecast",
    "description": "Get the weather forecast for a specific date, use this to determine an answer to questions like 'will it rain tomorrow?'",
    "input_schema": {
        "type": "object",
        "properties": {
            "date": {
                "type": "string",
                "description": "The forecast date",
            }
        },
        "required": ["date"]
    }
}

search_tool_claude = {
    "type":"custom",
    "name": "search_duckduckgo",
    "description": "Search the internet for information on a given query",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up information for"
            },
            "num_results": {
                "type": "integer",
                "description": "Number of search results to return (default: 5)",
            }
        },
        "required": ["query"]
    }
}

weather_tool_gemini = {
    'function_declarations':[{
        "name": "get_weather_forecast",
        "description": "Get the weather forecast for a specific date, use this to determine an answer to questions like 'will it rain tomorrow?'",
        "parameters": {
            "type":"object",
            "properties":{
                "date": {
                    "type": "string",
                    "description": "The forecast date",
                }
            },
            "required": ["date"]
        }
    }]
}

search_tool_gemini = {
    "function_declarations": [
        {
            "name": "search_duckduckgo",
            "description": "Search the internet for information on a given query",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query to look up information for"
                    },
                    "num_results": {
                        "type": "integer",
                        "description": "Number of search results to return (default: 5)",
                    }
                },
                "required": ["query"]
            }
        }
    ]
}

In [9]:
tools_openai = [{"type": "function", "function": weather_tool_openai}, {"type":"function", "function":search_tool_openai}]
tools_claude = [weather_tool_claude, search_tool_claude]
tools_gemini = [weather_tool_gemini,search_tool_gemini]

In [10]:
# HANDLE THE TOOL CALL FROM THE LLM - WITH DEBUGGING
def handle_tool_call(tool_call_dict):
    #print(f"DEBUG: handle_tool_call received: {tool_call_dict}")
    
    try:
        # Extract function info
        if 'function' not in tool_call_dict:
            error_msg = "Error: No 'function' key in tool call"
            #print(f"DEBUG: {error_msg}")
            return {
                "role": "tool",
                "tool_call_id": tool_call_dict.get('id', 'unknown'),
                "content": json.dumps({"error": error_msg})
            }
        
        function_name = tool_call_dict['function'].get('name', '')
        raw_args = tool_call_dict['function'].get('arguments', '')
        tool_call_id = tool_call_dict.get('id', 'unknown')
        
        #print(f"DEBUG: Function: {function_name}, Args: '{raw_args}', ID: {tool_call_id}")
        
        # Validate arguments
        if not raw_args or raw_args.strip() == '':
            error_msg = f"Error: Empty arguments for function {function_name}"
            #print(f"DEBUG: {error_msg}")
            return {
                "role": "tool",
                "tool_call_id": tool_call_id,
                "content": json.dumps({"error": error_msg})
            }
        
        # Parse arguments
        try:
            arguments = json.loads(raw_args)
            #print(f"DEBUG: Parsed arguments: {arguments}")
        except json.JSONDecodeError as e:
            error_msg = f"Error parsing JSON arguments: {e}"
            #print(f"DEBUG: {error_msg}")
            return {
                "role": "tool",
                "tool_call_id": tool_call_id,
                "content": json.dumps({"error": error_msg})
            }
        
        # Handle specific function calls
        if function_name == 'get_weather_forecast':
            date = arguments.get('date')
            if not date:
                error_msg = "Error: No date provided in arguments"
                #print(f"DEBUG: {error_msg}")
                return {
                    "role": "tool",
                    "tool_call_id": tool_call_id,
                    "content": json.dumps({"error": error_msg})
                }
            
            #print(f"DEBUG: Calling get_weather_forecast with date: {date}")
            forecast = get_weather_forecast(date)
            #print(f"DEBUG: Weather forecast result: {forecast}")
            
            return {
                "role": "tool",
                "tool_call_id": tool_call_id,
                "content": json.dumps({"forecast": forecast})
            }
        elif function_name == 'search_duckduckgo':
            query = arguments.get('query')
            if query:
                if arguments.get('num_results'):
                    num_results = arguments.get('num_results')
                    results = search_duckduckgo(query, num_results)
                    return{
                        "role":"tool",
                        "tool_call_id":tool_call_id,
                        "content":json.dumps({"results": results})
                    }
                else:
                    results = search_duckduckgo(query)
                    return{
                        "role":"tool",
                        "tool_call_id":tool_call_id,
                        "content":json.dumps({"results": results})
                    }
            if not query:
                error_msg = 'Error: No query provided in arguments'
                return{
                    "role":"tool",
                    "tool_call_id":tool_call_id,
                    "content":json.dumps({"error": error_msg})
                }
        else:
            error_msg = f"Error: Unknown function {function_name}"
            #print(f"DEBUG: {error_msg}")
            return {
                "role": "tool",
                "tool_call_id": tool_call_id,
                "content": json.dumps({"error": error_msg})
            }
            
    except Exception as e:
        error_msg = f"Error in handle_tool_call: {str(e)}"
        #print(f"DEBUG: {error_msg}")
        #import traceback
        #print(f"DEBUG: Traceback: {traceback.format_exc()}")
        return {
            "role": "tool",
            "tool_call_id": tool_call_dict.get('id', 'unknown'),
            "content": json.dumps({"error": error_msg})
        }

In [79]:
# OPENAI CHAT
def chat_openai(message, history):
    try:
        messages = [{"role": "system", "content": system_message}]
        messages.extend(history)
        messages.append({"role": "user", "content": message})
        #print(f"DEBUG: Sending messages to OpenAI: {messages}")
    
        stream = openai.chat.completions.create(
            model='gpt-4o-mini',
            messages=messages,
            tools=tools_openai,
            stream=True
        )
    
        response = ""
        tool_call_chunks = {}
        tool_call_detected = False
        
        # Generator function to yield content and update history
        chunk_id = ""
        for chunk in stream:
            #print(f"DEBUG: Raw chunk: {chunk}")
            choice = chunk.choices[0]
            #print(f"DEBUG: Choice: {choice}")
            #print(f"DEBUG: Choice.delta: {choice.delta}")
            
            # Handle tool calls
            if choice.delta.tool_calls:
                #print(f"DEBUG: Tool call chunk received: {choice.delta.tool_calls}")
                tool_call_detected = True
                
                for tc in choice.delta.tool_calls:
                    #print(f"DEBUG: Raw tool call object: {tc}")
                    #print(f"DEBUG: Tool call ID: {tc.id}")
                    #print(f"DEBUG: Tool call type: {tc.type}")
                    #print(f"DEBUG: Tool call function: {tc.function}")
                    
                    #if tc.function:
                    #    print(f"DEBUG: Function name: {repr(tc.function.name)}")
                    #    print(f"DEBUG: Function arguments: {repr(tc.function.arguments)}")
                    #    print(f"DEBUG: Function arguments type: {type(tc.function.arguments)}")
                    
                    # Initialize tool call if not exists
                    #chunk_id = ""
                    if tc.id is not None:
                        chunk_id = tc.id
                    #print(f'DEBUG: Chunk id is {chunk_id} and tc.id is {tc.id}')
                    if tc.id not in tool_call_chunks:
                        tool_call_chunks[tc.id] = {
                            "id": chunk_id,
                            "function": {
                                "name": "",
                                "arguments": ""
                            },
                            "type": tc.type or "function"
                        }
                        #print(f"DEBUG: Initialized new tool call chunk for ID: {chunk_id}")
                        
                    # Accumulate function name
                    if tc.function and tc.function.name is not None:
                        tool_call_chunks[chunk_id]["function"]["name"] += tc.function.name
                        #print(f"DEBUG: Function name chunk: id = {chunk_id}, fn = {repr(tc.function.name)}, total so far: {repr(tool_call_chunks[tc.id]['function']['name'])}")
                    
                    # Accumulate function arguments - this is the key fix
                    if tc.function and tc.function.arguments is not None:
                        #print(f"DEBUG: Adding arguments chunk: id = {chunk_id}, args = {repr(tc.function.arguments)} (length: {len(tc.function.arguments)})")
                        tool_call_chunks[chunk_id]["function"]["arguments"] += tc.function.arguments
                        #print(f"DEBUG: Arguments total so far: {repr(tool_call_chunks[chunk_id]['function']['arguments'])} (length: {len(tool_call_chunks[chunk_id]['function']['arguments'])})")
                    #else:
                    #    print(f"DEBUG: No arguments in this chunk - tc.function: {tc.function}")
                    #    if tc.function:
                    #        print(f"DEBUG: tc.function.arguments is: {repr(tc.function.arguments)}")
                    
                    #print(f"DEBUG: Current tool_call_chunks state: {tool_call_chunks}")
            
            # Handle regular content
            elif choice.delta.content:
                delta = choice.delta.content
                response += delta
                if delta:
                    yield response  # stream to Gradio frontend
            
            # Check if this is the end of the stream
            #if choice.finish_reason:
            #    print(f"DEBUG: Stream finished with reason: {choice.finish_reason}")
                
        # END FOR CHUNK IN STREAM LOOP
        
        # Update the history list
        history.append({"role": "user", "content": message})

        #print(f"DEBUG: Response is: {response}")
        #talker(response)
        
        # If no tool call was detected, just add the response and return
        if not tool_call_detected:
            print(f'DEBUG: Tool call not detected')
            if response:
                history.append({"role": "assistant", "content": response})
                print(f'DEBUG: No tool call response is {response}')
                talker(response)
            return
    
        # Tool call handling (post-stream)  
        if tool_call_detected:
            #print(f"DEBUG: Final tool_call_chunks: {tool_call_chunks}")
            
            if not tool_call_chunks:
                yield "Error: Tool call detected but no tool calls found"
                return
            
            # Convert tool_call_chunks to list for assistant message
            tool_calls_list = []
            tool_responses_list = []

            #print (f'DEBUG: Looping through tool_call_chunks:\n{tool_call_chunks}\n')
            
            # Process each tool call
            for tool_call_id, tool_call in tool_call_chunks.items():
                if not(tool_call['function']['name'] is None or (tool_call['function']['name']).strip() == ""):
                    #print(f"DEBUG: Processing tool call: {tool_call}")
                    
                    # Validate tool call completeness
                    function_name = tool_call["function"]["name"]
                    function_args = tool_call["function"]["arguments"]
    
                    #print(f'DEBUG: function_args are {function_args}')
                    
                    if not function_name:
                        yield f"Error: Tool call missing function name"
                        return
                        
                    #if not function_args or function_args.strip() == "":
                        # Fallback: Try non-streaming approach
                        #print(f"DEBUG: Arguments empty, trying non-streaming fallback...")
                    
                    # Validate JSON arguments
                    try:
                        json.loads(function_args)
                    except json.JSONDecodeError as e:
                        yield f"Error: Invalid JSON in tool call arguments: {e}"
                        return
                    
                    # Add to tool calls list
                    tool_calls_list.append(tool_call)
                    #print(f'DEBUG: Tool call is {tool_call}')
                    
                    # Handle the tool call
                    tool_response = handle_tool_call(tool_call)
                    #print(f"DEBUG: Tool response: {tool_response}")
                    #print(f'DEBUG: JSON tool response content: {tool_response['content']}')

                    #Add to tool responses list
                    tool_responses_list.append({
                        "role":"tool",
                        "tool_call_id":tool_response['tool_call_id'],
                        "content": json.dumps({'forecast': json.loads(tool_response['content'])})
                    })
                    
                    print(f'DEBUG: Tool response list is {tool_responses_list}')
                # END IF FUNCTION NAME TEST

            # END TOOL_CALLS FOR LOOP
            # Add tool response to messages
            if tool_calls_list:
                messages.append({
                    "role": "assistant", 
                    "content": None,  # Important: set to None when there are tool calls
                    "tool_calls": tool_calls_list
                })

            if tool_responses_list:
                messages.extend(tool_responses_list)

        #print(f"DEBUG: Messages before final completion: {messages}")
    
        # Get final response from assistant
        final_completion = openai.chat.completions.create(
            model='gpt-4o-mini',
            messages=messages,
            temperature=0.7
        )
        
        final_response = final_completion.choices[0].message.content
        #print(f"DEBUG: Final response: {final_response}")
        
        if final_response:
            yield final_response
            history.append({"role": "assistant", "content": final_response})
            #print(f'DEBUG: Calling talker')
            talker(final_response)
        elif tool_responses_list:
            for r in tool_responses_list:
                    #print(f'DEBUG: Tool response entry content is {json.loads(r['content'])['forecast']['forecast']}')
                    talker(json.dumps(json.loads(r['content'])['forecast']))
            
    except Exception as e:
        import traceback
        error_msg = f'Error in chat_openai: {str(e)}\n{traceback.format_exc()}'
        print(error_msg)
        yield error_msg

In [80]:
def chat_claude(message, history):
    result = claude.messages.stream(
        model="claude-opus-4-20250514",
        max_tokens = 1000,
        temperature = 0.4,
        system = system_message,
        tools = tools_claude,
        messages = history + [{"role":"user","content":message}]
    )
    response = ""
    tool_calls = []
    tool_responses = []
    tool_call_id = ""
    tool_call_fn = ""
    tool_call_args = ""
    tool_call_detected = False
    tool_call_item = None
    text_content = ""  # Track text content separately

    with result as stream:
        for chunk in stream:
            if chunk.type == 'content_block_start' and chunk.content_block.type == 'tool_use':
                tool_call_detected = True
                tool_call_id = chunk.content_block.id
                tool_call_fn = chunk.content_block.name
                tool_call_item = {
                        "id":chunk.content_block.id,
                        "function":{
                            "name":chunk.content_block.name,
                            "arguments":""
                        },
                        "type":"function"
                    }

                response += f'...using tool {tool_call_fn}...'
                yield response

            elif chunk.type == 'content_block_delta':
                if hasattr(chunk.delta, 'text'):
                    text = chunk.delta.text
                    text_content += text  # Track actual text content
                    response += text
                    yield response
                # GET THE TOOL ARGUMENTS IF ANY
                elif hasattr(chunk.delta, 'partial_json') and tool_call_item:
                    tool_call_item['function']['arguments'] += chunk.delta.partial_json
                    
            elif chunk.type == 'content_block_stop' and tool_call_detected:
                if hasattr(chunk, 'content_block') and chunk.content_block.type == 'tool_use':
                    if tool_call_item:
                        try:
                            tool_calls.append(tool_call_item)
                            tool_response = handle_tool_call(tool_call_item)
                            
                            # handle_tool_call returns OpenAI format: {"role": "tool", "tool_call_id": "...", "content": "..."}
                            # We need to extract just the content for Anthropic API
                            if isinstance(tool_response, dict) and 'content' in tool_response:
                                # Extract the content from the OpenAI-style response
                                tool_content = tool_response['content']
                            else:
                                # Fallback if format is different
                                tool_content = json.dumps(tool_response) if isinstance(tool_response, dict) else str(tool_response)
                            
                            # Store tool response with proper format for Anthropic API
                            tool_responses.append({
                                "tool_call_id": tool_call_item['id'],
                                "content": tool_content
                            })

                            # Don't add raw tool output to response or yield it
                            # The follow-up request will handle the proper formatting
                        except Exception as e:
                            print(f'ERROR: {str(e)}')
                            err_msg = f'ERROR: Tool error - {str(e)}'
                            # Don't add error to response either, let follow-up handle it
                            tool_responses.append({
                                "tool_call_id": tool_call_item['id'],
                                "content": json.dumps({"error": err_msg})
                            })

                        # Reset for next potential tool call
                        tool_call_item = None
                        tool_call_detected = False

    if tool_responses:
        print(f'DEBUG: Making followup request with tool response')

        # Build proper assistant message with both text and tool calls
        assistant_content = []
        
        # Add text content if any
        if text_content.strip():
            assistant_content.append({
                "type": "text",
                "text": text_content
            })
        
        # Add tool calls
        for tool_call in tool_calls:
            assistant_content.append({
                "type": "tool_use",
                "id": tool_call["id"],
                "name": tool_call["function"]["name"],
                "input": json.loads(tool_call["function"]["arguments"]) if tool_call["function"]["arguments"] else {}
            })

        # Build followup messages with proper tool result format
        # Tool results must be in a user message, not assistant message
        user_content_with_tools = []
        for tool_response in tool_responses:
            user_content_with_tools.append({
                "type": "tool_result",
                "tool_use_id": tool_response["tool_call_id"],
                "content": tool_response["content"]
            })
        
        followup_messages = history + [
            {"role":"user", "content":[{"type": "text", "text": message}]},
            {"role":"assistant", "content": assistant_content},
            {"role":"user", "content": user_content_with_tools + [
                {"type": "text", "text": "Based on the search results above, please provide a comprehensive answer to my question. Do not search for additional information."}
            ]}
        ]

        print(f'DEBUG: Followup messages - {followup_messages}')

        # Reset response to only show the formatted response, not the raw tool output
        follow_up_response = ""
        
        follow_up_result = claude.messages.stream(
            model="claude-opus-4-20250514",
            max_tokens=1000,
            temperature=0.4,
            system=system_message,
            # Don't include tools in follow-up to prevent recursive tool calls
            # tools=tools_claude,
            messages=followup_messages
        )
        
        print(f'DEBUG: Followup result is type {type(follow_up_result)}')
        print(f'DEBUG: Has __iter__: {hasattr(follow_up_result, "__iter__")}')

        try:
            with follow_up_result as follow_up_stream:
                for chunk in follow_up_stream:
                    if chunk.type == 'content_block_delta' and hasattr(chunk.delta, 'text'):
                        text = chunk.delta.text
                        follow_up_response += text
                        print(f'DEBUG: Followup total response is - {follow_up_response}')
                        yield follow_up_response
        except Exception as e:
            print(f"Error iterating over stream: {e}")
            print(f"Stream object: {follow_up_result}")
        
        # Use the follow-up response as the final response
        final_response = follow_up_response
    else:
        # No tool calls, use the original response
        final_response = response
    
    # Add final message to history with the correct content
    history.append({"role":"user", "content":[{"type": "text", "text": message}]})
    history.append({"role":"assistant", "content":[{"type": "text", "text": final_response}]})
    
    yield final_response

In [81]:
def chat_gemini(message, history):
    """
    Stream chat responses from Google Gemini API with tool calling support.
    Converts the Claude streaming pattern to work with Gemini.
    """
    
    # Initialize model with tools
    model = genai.GenerativeModel(
        model_name="gemini-2.5-flash-lite",
        tools=tools_gemini,  # Your tools list converted to Gemini format
        system_instruction=system_message
    )
    
    # Convert history to Gemini format
    gemini_history = []
    for msg in history:
        if msg["role"] == "user":
            gemini_history.append({"role": "user", "parts": [msg["content"]]})
        elif msg["role"] == "assistant":
            gemini_history.append({"role": "model", "parts": [msg["content"]]})
        elif msg["role"] == "tool":
            # Handle tool responses in history
            gemini_history.append({
                "role": "function",
                "parts": [{"function_response": {"name": "tool_response", "response": msg["content"]}}]
            })
    
    # Add current message
    gemini_history.append({"role": "user", "parts": [message]})
    
    response = ""
    tool_calls = []
    tool_responses = []
    tool_call_detected = False
    
    try:
        # Generate streaming response
        stream_response = model.generate_content(
            gemini_history,
            generation_config=genai.types.GenerationConfig(
                max_output_tokens=1000,
                temperature=0.4,
            ),
            stream=True
        )
        
        for chunk in stream_response:
            print(f'DEBUG: Current chunk - {chunk}')

            print(f'DEBUG: Looking for tool calls')
            # Handle function calls (tool calls)
            if chunk.candidates and len(chunk.candidates) > 0:
                candidate = chunk.candidates[0]
                if candidate.content and candidate.content.parts:
                    for part in candidate.content.parts:
                        print(f'DEBUG: Current part is {part}')
                        if hasattr(part, 'function_call') and part.function_call:
                            tool_call_detected = True
                            print(f'DEBUG: Part function call - {part.function_call}')
                            function_call = part.function_call
                            
                            print(f'DEBUG: Gemini detected a tool call : {function_call}')
                            
                            # Convert Gemini function call to Claude-like format
                            tool_call_item = {
                                "id": f"call_{len(tool_calls)}",  # Generate ID
                                "function": {
                                    "name": function_call.name,
                                    "arguments": json.dumps(dict(function_call.args))
                                },
                                "type": "function"
                            }
                            
                            tool_calls.append(tool_call_item)
                            print(f'DEBUG: Tool call item definition is {tool_call_item}')
                            
                            response += f'...using tool {function_call.name}...'
                            yield response
                            
                            try:
                                print(f'DEBUG: Calling tool {function_call.name}')
                                tool_response = handle_tool_call(tool_call_item)
                                tool_responses.append({
                                    "tool_call_id": tool_call_item['id'],
                                    "role": "tool",
                                    "content": tool_response
                                })
                                print(f'DEBUG: Tool response is - {tool_response}')

                                print(f"DEBUG: Response prior - {response}")
                                response += json.dumps(tool_response)
                                print(f"DEBUG: Response after - {response}")
                                yield response
                                
                            except Exception as e:
                                print(f'ERROR: {str(e)}')
                                err_msg = f'ERROR: Tool error - {str(e)}'
                                response += err_msg
                                yield response
                                
                    # END FOR LOOP

                        else: # IF part.function_call ELSE
                            try:
                                print(f'DEBUG: Handling chunk text')
                                # Handle text content
                                if hasattr(chunk, 'text') and chunk.text:
                                    response += chunk.text
                                    print(F"DEBUG:  Response after adding chunk text - {response}")
                                    yield response
                            except Exception as e:
                                print(f'DEBUG: ERROR - {str(e)}')
        
        #print(f'DEBUG: Total response is - {response}')
        #print(f'DEBUG: All tool calls - {str(tool_calls)}')
        #print(f'DEBUG: All tool responses - {tool_responses}')
        
        # Handle follow-up request if tools were called
        if tool_responses:
            print(f'DEBUG: Making followup request with tool response')
            
            # Build follow-up history with tool responses
            followup_history = gemini_history.copy()
            
            # Add the assistant's response with tool calls
            print("DEBUG: Add assistant's response with tool calls")
            assistant_parts = []
            print(f"DEBUG: current response is {response}")
            if response.replace(''.join([f'...using tool {tc["function"]["name"]}...' + json.dumps(tr["content"]) for tc, tr in zip(tool_calls, tool_responses)]), '').strip():
                assistant_parts.append(response.split('...using tool')[0])
            
            # Add function calls to the assistant message
            print("DEBUG: Add function calls to the assistant message")
            for i, tool_call in enumerate(tool_calls):
                function_call_part = genai.protos.Part(
                    function_call=genai.protos.FunctionCall(
                        name=tool_call["function"]["name"],
                        args=json.loads(tool_call["function"]["arguments"])
                    )
                )
                assistant_parts.append(function_call_part)
            
            if assistant_parts:
                followup_history.append({"role": "model", "parts": assistant_parts})
            
            # Add function responses
            print("DEBUG: Add function responses")
            for tool_response in tool_responses:
                followup_history.append({
                    "role": "function",
                    "parts": [{
                        "function_response": {
                            "name": next(tc["function"]["name"] for tc in tool_calls if tc["id"] == tool_response["tool_call_id"]),
                            "response": {"result": tool_response["content"]}
                        }
                    }]
                })
            
            # Generate follow-up response
            follow_up_result = model.generate_content(
                followup_history,
                generation_config=genai.types.GenerationConfig(
                    max_output_tokens=1000,
                    temperature=0.4,
                ),
                stream=True
            )
            
            response += "\n\n"
            yield response
            
            for chunk in follow_up_result:
                if chunk.text:
                    response += chunk.text
                    yield response
        print(f'DEBUG: Final response is {response}')
    
    except Exception as e:
        print(f'ERROR: Stream error - {str(e)}')
        error_msg = f'ERROR: {str(e)}'
        response += error_msg
        yield response

    print(f'DEBUG: Updating history')
    # Update history
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response})
    
    yield response


In [82]:
def stream_chat(message, model):
    if model == 'OpenAI':
        yield from chat_openai(message, history)
    elif model == 'Claude':
        yield from chat_claude(message, history)
    elif model == 'Gemini':
        yield from chat_gemini(message, history)
    else:
        raise ValueError('Sorry we don''t support that model yet!')

In [83]:
#talker('Hello, i am a talker.')

In [84]:
history = []
total_response = ""
for r in stream_chat("What day in the next 3 days is best for a hike?", 'OpenAI'):
    total_response = r

#talker(total_response)

DEBUG: Tool response list is [{'role': 'tool', 'tool_call_id': 'call_EpVVe9VATvAosnMTwcybZWii', 'content': '{"forecast": {"forecast": "A slight chance of showers and thunderstorms before 7pm. Mostly clear, with a low around 74. Heat index values as high as 99. East wind 0 to 5 mph. Chance of precipitation is 20%."}}'}]


In [86]:
history = []
view = gr.Interface(
    fn=stream_chat,
    inputs=[gr.Textbox(label="Message:"), gr.Dropdown(["OpenAI","Claude","Gemini"], label="Model:")],
    outputs=[gr.Markdown(label="Response:")],
    flagging_mode="never"
)
view.launch()
#gr.ChatInterface(fn=chat_openai, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
